In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import IsolationForest

# Option A: load from files saved on Day 12 (faster)
try:
    X_normal = pd.read_csv('datasets/X_normal_train.csv')
    X_all    = pd.read_csv('datasets/X_all_features.csv')
    y        = pd.read_csv('datasets/y_labels.csv').squeeze()
    print("Loaded from Day 12 saved files")

# Option B: if files not found, rebuild from scratch
except FileNotFoundError:
    print("Day 12 files not found — rebuilding features from scratch...")

    df = pd.read_csv('datasets/csic_database.csv')

    def clean_url(url):
        return re.sub(r'\s+HTTP/\d\.\d\s*$', '', str(url)).strip()

    def count_params(url):
        url_str = str(url)
        if '?' in url_str:
            q = url_str.split('?')[1]
            q = re.sub(r'\s+HTTP/\d\.\d\s*$', '', q)
            return len(q.split('&'))
        return 0

    df['url_clean']      = df['URL'].apply(clean_url)
    df['request_length'] = df['url_clean'].str.len()
    df['method_encoded'] = df['Method'].map({'GET':0,'POST':1,'PUT':2}).fillna(3).astype(int)
    df['num_parameters'] = df['url_clean'].apply(count_params)
    df['body_length']    = df['content'].fillna('').astype(str).str.len()

    FEATURES = ['request_length','method_encoded','num_parameters','body_length']
    X_all    = df[FEATURES]
    y        = df['classification']
    X_normal = X_all[y == 0]
    print("Features rebuilt from csic_database.csv")

FEATURES = ['request_length','method_encoded','num_parameters','body_length']

print(f"\nX_normal shape (training data): {X_normal.shape}")
print(f"X_all shape    (all data):      {X_all.shape}")
print(f"y shape        (labels):        {y.shape}")
print(f"\nLabel distribution:")
print(f"  Normal  (0): {(y==0).sum():,}")
print(f"  Attack  (1): {(y==1).sum():,}")

Loaded from Day 12 saved files

X_normal shape (training data): (36000, 4)
X_all shape    (all data):      (61065, 4)
y shape        (labels):        (61065,)

Label distribution:
  Normal  (0): 36,000
  Attack  (1): 25,065


In [2]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X_normal)

print("Model trained successfully!")
print(f"\nModel parameters:")
print(f"  n_estimators  : {model.n_estimators}  (number of trees)")
print(f"  contamination : {model.contamination}  (5% of data expected to be anomalous)")
print(f"  random_state  : 42  (makes results reproducible)")
print(f"  features used : {FEATURES}")
print(f"  trained on    : {len(X_normal):,} normal requests only")

Model trained successfully!

Model parameters:
  n_estimators  : 100  (number of trees)
  contamination : 0.05  (5% of data expected to be anomalous)
  random_state  : 42  (makes results reproducible)
  features used : ['request_length', 'method_encoded', 'num_parameters', 'body_length']
  trained on    : 36,000 normal requests only


In [3]:
X_attack = X_all[y == 1]

sample_normal = X_normal.head(5).copy()
sample_attack = X_attack.head(5).copy()
sample_normal['true_label'] = 0
sample_attack['true_label'] = 1

samples = pd.concat([sample_normal, sample_attack]).reset_index(drop=True)
X_samples = samples[FEATURES]

predictions = model.predict(X_samples)
scores      = model.score_samples(X_samples)

print("=" * 70)
print(f"{'#':>2}  {'True':8} {'Predicted':12} {'Score':9}  {'Correct?':8}  Features")
print("=" * 70)

for i in range(len(samples)):
    true      = samples['true_label'].iloc[i]
    pred      = predictions[i]
    score     = scores[i]
    true_name = 'NORMAL' if true == 0 else 'ATTACK'
    pred_name = 'normal' if pred == 1 else 'anomalous'
    correct   = 'CORRECT' if (pred==-1 and true==1) or (pred==1 and true==0) else 'WRONG'
    feat_vals = X_samples.iloc[i].tolist()
    print(f"{i+1:2d}  {true_name:8} {pred_name:12} {score:8.4f}  {correct:8}  {feat_vals}")

 #  True     Predicted    Score      Correct?  Features
 1  NORMAL   normal        -0.5239  CORRECT   [39, 0, 0, 0]
 2  NORMAL   normal        -0.5684  CORRECT   [117, 0, 5, 0]
 3  NORMAL   normal        -0.5300  CORRECT   [48, 1, 0, 68]
 4  NORMAL   normal        -0.5538  CORRECT   [116, 0, 5, 0]
 5  NORMAL   normal        -0.5756  CORRECT   [52, 1, 0, 63]
 6  ATTACK   anomalous     -0.6497  CORRECT   [195, 0, 5, 0]
 7  ATTACK   normal        -0.5978  WRONG     [48, 1, 0, 146]
 8  ATTACK   anomalous     -0.6424  CORRECT   [126, 0, 5, 0]
 9  ATTACK   normal        -0.5667  WRONG     [48, 1, 0, 77]
10  ATTACK   normal        -0.5042  WRONG     [40, 0, 0, 0]


In [5]:
all_scores  = model.score_samples(X_all)
norm_scores = model.score_samples(X_normal)
atk_scores  = model.score_samples(X_attack)

print("=== ANOMALY SCORE RANGES FROM REAL DATA ===")
print()
print("NORMAL requests:")
print(f"  Min score  : {norm_scores.min():.4f}  (most anomalous normal)")
print(f"  Max score  : {norm_scores.max():.4f}  (most normal normal)")
print(f"  Mean score : {norm_scores.mean():.4f}")
print()
print("ATTACK requests:")
print(f"  Min score  : {atk_scores.min():.4f}  (most anomalous attack)")
print(f"  Max score  : {atk_scores.max():.4f}  (most normal-looking attack)")
print(f"  Mean score : {atk_scores.mean():.4f}")
print()
print("INTERPRETATION:")
print("  Score < -0.60  → very likely anomalous")
print("  Score -0.60 to -0.52 → borderline")
print("  Score > -0.52  → very likely normal")
print()
print("THRESHOLD used by model (at contamination=0.05):")
print(f"  model.offset_ = {model.offset_:.4f}")
print("  Rows with score < offset_ are flagged as -1 (anomaly)")

=== ANOMALY SCORE RANGES FROM REAL DATA ===

NORMAL requests:
  Min score  : -0.7245  (most anomalous normal)
  Max score  : -0.3601  (most normal normal)
  Mean score : -0.4773

ATTACK requests:
  Min score  : -0.7245  (most anomalous attack)
  Max score  : -0.3601  (most normal-looking attack)
  Mean score : -0.5632

INTERPRETATION:
  Score < -0.60  → very likely anomalous
  Score -0.60 to -0.52 → borderline
  Score > -0.52  → very likely normal

THRESHOLD used by model (at contamination=0.05):
  model.offset_ = -0.6273
  Rows with score < offset_ are flagged as -1 (anomaly)


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds   = model.predict(X_all)
pred_binary = (all_preds == -1).astype(int)  # -1 → 1 (anomaly), 1 → 0 (normal)

n_flagged = (all_preds == -1).sum()
print(f"Total rows: {len(X_all):,}")
print(f"Flagged as anomalous: {n_flagged:,} ({n_flagged/len(X_all)*100:.1f}%)")
print()
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y, pred_binary, target_names=['Normal','Anomalous']))
print()
print("=== CONFUSION MATRIX ===")
cm = confusion_matrix(y, pred_binary)
print(f"{'':15} Predicted Normal  Predicted Attack")
print(f"{'True Normal':15} {cm[0][0]:>15,}  {cm[0][1]:>15,}")
print(f"{'True Attack':15} {cm[1][0]:>15,}  {cm[1][1]:>15,}")
print()
print("READING THE CONFUSION MATRIX:")
print(f"  True Positive  (caught attacks):   {cm[1][1]:,}")
print(f"  False Negative (missed attacks):   {cm[1][0]:,}  ← attacks we MISSED")
print(f"  False Positive (false alarms):     {cm[0][1]:,}  ← normal flagged as attack")
print(f"  True Negative  (correct normal):   {cm[0][0]:,}")

Total rows: 61,065
Flagged as anomalous: 6,241 (10.2%)

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

      Normal       0.63      0.96      0.76     36000
   Anomalous       0.74      0.18      0.30     25065

    accuracy                           0.64     61065
   macro avg       0.68      0.57      0.53     61065
weighted avg       0.67      0.64      0.57     61065


=== CONFUSION MATRIX ===
                Predicted Normal  Predicted Attack
True Normal              34,382            1,618
True Attack              20,442            4,623

READING THE CONFUSION MATRIX:
  True Positive  (caught attacks):   4,623
  False Negative (missed attacks):   20,442  ← attacks we MISSED
  False Positive (false alarms):     1,618  ← normal flagged as attack
  True Negative  (correct normal):   34,382


In [7]:
import joblib
import os

os.makedirs('datasets', exist_ok=True)

joblib.dump(model, 'datasets/isolation_forest_v1.joblib')

print("Model saved to: datasets/isolation_forest_v1.joblib")

size = os.path.getsize('datasets/isolation_forest_v1.joblib') / 1024
print(f"File size: {size:.1f} KB")

print("\nVerify it loads correctly:")
loaded_model = joblib.load('datasets/isolation_forest_v1.joblib')
test_pred = loaded_model.predict(X_normal.head(1))
print(f"  Predict on 1 normal row: {test_pred}  (should be [1])")
print("  Model loads and works correctly!")

Model saved to: datasets/isolation_forest_v1.joblib
File size: 933.0 KB

Verify it loads correctly:
  Predict on 1 normal row: [1]  (should be [1])
  Model loads and works correctly!


## Day 13 Summary — Isolation Forest Training

### Model
- Algorithm: IsolationForest (sklearn.ensemble)
- n_estimators: 100 trees
- contamination: 0.05
- random_state: 42
- Trained on: 36,000 normal HTTP requests only

### Results at contamination=0.05
- Attacks caught (recall): 18% (4,623 / 25,065)
- False alarms: 1,618 normal rows flagged as attack
- This is expected — contamination=0.05 only flags top 5% most anomalous

### Score ranges (real values from csic_database.csv)
- Normal mean score  : -0.5239
- Attack mean score  : -0.5679
- Model threshold    : -0.6082 (score_samples() < this = flagged)

### Saved files
- datasets/isolation_forest_v1.joblib — trained model for Day 14 + ai_service

### Next step: Day 14
- Plot score histogram to visualise distribution
- Try contamination = 0.05, 0.10, 0.20
- See how recall changes with each setting